In [1]:
# -*- coding: utf-8 -*-
import torch
import torch.utils.data as data
import torchmetrics
import numpy as np
from scipy.stats import pearsonr
from tqdm import tqdm

from model.SCRN import SCRN
from util.dataset import SeismicHDF5Dataset
from util.My_tool1 import snr_, ssim_

/home/SUZummer/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
TEST_PATH = 'inputs/data/test-4-128.hdf5'
MODEL_PATH = 'trained_model/best_train_model_second_data.pth'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 24

total_snr, total_ssim, total_res_corr, num_samples = 0, 0, 0, 0

In [3]:
d_test = SeismicHDF5Dataset(TEST_PATH)
test_data = data.DataLoader(d_test, batch_size=batch_size)

mse = torchmetrics.MeanSquaredError().to(device)
mae = torchmetrics.MeanAbsoluteError().to(device)
r2 = torchmetrics.R2Score().to(device)

In [4]:
model = SCRN().to(device)
state_dict = torch.load(MODEL_PATH)
model.load_state_dict(state_dict)
model.eval()
print()

In [6]:
with torch.no_grad():
    tqdm_data = tqdm(test_data, leave=True)
    for x, y in tqdm_data:
        x, y = x.to(device), y.to(device)
        pred = model(x)

        pred_flat = pred.view(pred.size(0), -1)
        y_flat = y.view(y.size(0), -1)

        mse.update(pred, y)
        mae.update(pred, y)
        r2.update(pred_flat, y_flat)

        batch_size = x.size(0)
        num_samples += batch_size

        pred_np = pred.cpu().numpy().squeeze(1)
        y_np = y.cpu().numpy().squeeze(1)

        for i in range(pred_np.shape[0]):
            pred_i = pred_np[i]
            y_i = y_np[i]

            total_snr += snr_(pred_i, y_i)
            total_ssim += ssim_(pred_i, y_i)

            pred_i_flat = pred_i.flatten()
            y_i_flat = y_i.flatten()
            
            res_corr = y_i_flat - pred_i_flat
            if np.std(res_corr) > 1e-8 and np.std(y_i_flat) > 1e-8:
                corr, _ = pearsonr(res_corr, y_i_flat)
                total_res_corr += corr
            else:
                total_res_corr += 0

100%|██████████| 956/956 [02:54<00:00,  5.47it/s]


In [7]:
avg_r2 = r2.compute().item()
avg_snr = total_snr / num_samples
avg_ssim = total_ssim / num_samples
avg_mse = mse.compute().item()
avg_rmse = torch.sqrt(torch.tensor(avg_mse)).item()
avg_mae = mae.compute().item()
avg_res_corr = total_res_corr / num_samples

In [8]:
print(f'Результат тестирования модели {MODEL_PATH}:')
print(f'  R²   = {avg_r2:.6f}')
print(f'  SNR  = {avg_snr:.6f} dB')
print(f'  SSIM = {avg_ssim:.6f}')
print(f'  RMSE = {avg_rmse:.6f}')
print(f'  MAE  = {avg_mae:.6f}')
print(f'  Res. Corr. = {avg_res_corr:.6f}')

Результат тестирования модели trained_model/best_train_model_second_data.pth:
  R²   = 0.998049
  SNR  = 51.363384 dB
  SSIM = 0.996342
  RMSE = 0.003409
  MAE  = 0.000729
  Res. Corr. = 0.099908
